In [3]:
#!/usr/bin/env python3
"""
QPA-ising circuit simulation with simplified implementation.

This implementation uses:
- k=2 qubits per register
- Single control qubit
- Single QPA sequence (H, CSWAP, H)
- Two projectors for z=0 and z=1 states
"""
from itertools import product
import numpy as np
from qiskit import QuantumCircuit, transpile, ClassicalRegister, QuantumRegister
from qiskit.quantum_info import Statevector, Operator, SparsePauliOp
from qiskit_aer import AerSimulator
import os
from qiskit.visualization import circuit_drawer
import argparse
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
import pandas as pd
from scipy.linalg import expm
from dotenv import load_dotenv
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2, SamplerV2
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_aer.noise import (
    NoiseModel,
    QuantumError,
    ReadoutError,
    depolarizing_error,
    pauli_error,
    thermal_relaxation_error,
)

from tqdm import tqdm


def get_QPA_circuit(k,nqpa, LAMBDA):
    """
    Returns a simplified QPA circuit with single control qubit and single QPA sequence.
    
    Args:
        d: Number of qubits per register
        ising_circuit: ising_class instance
        
    Returns:
        QuantumCircuit implementing the QPA sequence
    """
    # Single control qubit
    if nqpa==0:
        cr = ClassicalRegister(k, name='control') #Will only Measure q3
    if nqpa==1:
        cr = ClassicalRegister(1+2*k, name='control')
    qr_all = QuantumRegister(3*k + 1)
    
    # Initialize quantum circuit
    qc = QuantumCircuit(qr_all, cr)

    rng = np.random.default_rng()

    reg_ranges = [range(1, 1 + k), range(1+ k, 1+2 * k), range(1+2 * k, 1+3 * k)]
    pauli_strs = ['i', 'x', 'y', 'z']

    for reg in reg_ranges:
        if rng.random() < LAMBDA:
            paulis = rng.choice(pauli_strs, size=k)
            for q, p in zip(reg, paulis):
                if p == 'x':
                    qc.x(q)
                elif p == 'y':
                    qc.y(q)
                elif p == 'z':
                    qc.z(q)
    
    if nqpa==0:
        for i in range(k): #Only measure Q3
            qc.measure(2 * k + i+1, cr[i])
        return qc

    assert nqpa==1 #Cannot have a different value for this circuit for now
    
    # Single QPA sequence
    qc.h(0)      # Apply Hadamard
    for i in range(k):
        qc.cswap(0, i+1, i+k+1)  # Control qubit 0, target q1 and q2
    qc.h(0)      # Apply second Hadamard

    qc.measure(0,cr[0])#Measure Q0
    for i in range(k): #Measure Q2
        qc.measure(1+k+i, cr[i+1])
    for i in range(k): #Measure Q3
        qc.measure(1+2*k+i,cr[i+1+k])
    return qc

In [6]:

# Instantiate QPA circuit
LAMBDA = 1
k = 2
nqpa = 1
qc = get_QPA_circuit(k, nqpa, LAMBDA)

# Load your IBM Quantum account (assumes you've already saved it)
service = QiskitRuntimeService()

# Get list of backends (filter to simulators=False if you only want real devices)
backends = service.backends(simulator=False, operational=True)

# Check which backends can support the circuit
compatible_backends = []

for backend in backends:
    try:
        qc_transpiled = transpile(qc, backend=backend,optimization_level=3)
        if backend.configuration().n_qubits >= qc_transpiled.num_qubits:
            compatible_backends.append((backend.name, backend.configuration().n_qubits))
    except Exception as e:
        # Skip backends that cannot transpile the circuit
        print(f"Backend {backend.name} skipped: {e}")
        continue
    

print("\n✅ Compatible backends:")
for name, qubits in compatible_backends:
    print(f"• {name} ({qubits} qubits)")


/tmp/ipykernel_1571238/2181288811.py:8: DeprecationWarning: The "ibm_quantum" channel option is deprecated and will be sunset on 1 July. After this date, "ibm_cloud", "ibm_quantum_platform", and "local" will be the only valid channels. Open Plan users should migrate now.  All other users should review the migration guide (https://quantum.cloud.ibm.com/docs/migration-guides/classic-iqp-to-cloud-iqp)to learn when to migrate.
  service = QiskitRuntimeService()



✅ Compatible backends:
• ibm_aachen (156 qubits)
• ibm_brisbane (127 qubits)
• ibm_brussels (127 qubits)
• ibm_fez (156 qubits)
• ibm_kingston (156 qubits)
• ibm_marrakesh (156 qubits)
• ibm_sherbrooke (127 qubits)
• ibm_strasbourg (127 qubits)
• ibm_torino (133 qubits)
